# Histology Backbone: Progressive_K5 3-Model Ensemble

Train a **3-model binary ensemble** with the same strategy and hyperparameters as the ImageNet progressive_K5 baseline, but using the **histology-pretrained backbone**. Then export to ONNX and evaluate on frozen sets **T5, TB, TM** (and TC if available).

**Matches baseline:** dropout [0.3, 0.1, 0.2], seeds base/base+1/base+2, warmup 5 ep, unfreeze last 5 blocks, lr_head 1e-3, lr_backbone 1e-4, 50 epochs, patience 12, same data and augmentation.

## 1. Configuration

In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime

NOTEBOOK_DIR = Path(".").resolve()
FROZEN_DIR = NOTEBOOK_DIR / "frozen_test_sets"
BACKBONE_CKPT = NOTEBOOK_DIR / "results" / "histology_pretrain" / "backbone_state_dict.pth"

DATA_DIR_5CLASS = os.path.expanduser("~/dev/oral_cavity_5class_dataset")
DATA_DIR_BINARY = os.path.expanduser("~/dev/oral_cavity_binary_dataset")
DATA_DIR_SMALL_BINARY = os.path.expanduser("~/dev/oral-images-dataset/Oral-Image-small-binary-dataset")

run_name = f"histology_progressive_K5_ensemble_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = NOTEBOOK_DIR / "results" / "histology_progressive_K5_ensemble" / run_name
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(exist_ok=True)

DROPOUT_RATES = [0.3, 0.1, 0.2]
ENSEMBLE_WEIGHTS = [0.30, 0.40, 0.30]

CONFIG = {
    "image_size": (224, 224),
    "batch_size": 32,
    "num_epochs": 50,
    "patience": 12,
    "val_split": 0.15,
    "num_workers": 0,
    "seed": 42,
    "augmentation": "medical",
    "cancer_weight_multiplier": 1.5,
    "label_smoothing": 0.05,
    "warmup_epochs": 5,
    "unfreeze_last_k": 5,
    "lr_head": 1e-3,
    "lr_backbone": 1e-4,
}

with open(OUTPUT_DIR / "config.json", "w") as f:
    json.dump({**CONFIG, "run_name": run_name, "backbone_ckpt": str(BACKBONE_CKPT), "dropout_rates": DROPOUT_RATES}, f, indent=2)

if not BACKBONE_CKPT.exists():
    raise FileNotFoundError(f"Run histology_pretrain.ipynb first. Missing: {BACKBONE_CKPT}")

print(f"Output: {OUTPUT_DIR}")
print(f"Dropout rates: {DROPOUT_RATES}")

Output: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_progressive_K5_ensemble/histology_progressive_K5_ensemble_20260219_083432
Dropout rates: [0.3, 0.1, 0.2]


## 2. Imports and device

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALB = True
except ImportError:
    HAS_ALB = False

try:
    from torch.utils.tensorboard import SummaryWriter
    HAS_TB = True
except ImportError:
    HAS_TB = False

sys.path.insert(0, str(NOTEBOOK_DIR))
from frozen_test_sets import build_and_save_frozen_sets_v2, get_all_train_val_for_training

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
NUM_CLASSES = 2

if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA")
else:
    device = torch.device("cpu")
    print("Using CPU")

Using MPS


## 3. Load frozen sets and merge training data

In [3]:
T5, TB, TC, TM, meta = build_and_save_frozen_sets_v2(
    data_dir_5class=DATA_DIR_5CLASS,
    data_dir_binary=DATA_DIR_BINARY,
    data_dir_small_binary=DATA_DIR_SMALL_BINARY,
    out_dir=FROZEN_DIR,
)

def collect_images(folder, exts=("*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG")):
    folder = Path(folder)
    out = []
    if folder.exists():
        for ext in exts:
            out.extend([str(p) for p in folder.glob(ext)])
    return out

binary_train_paths, binary_train_labels, _ = get_all_train_val_for_training(FROZEN_DIR)
five_dir = Path(DATA_DIR_5CLASS)
five_cancer, five_non = [], []
for split in ["train", "val"]:
    for c in ["OCA", "OPMD"]:
        five_cancer.extend(collect_images(five_dir / split / c))
    for c in ["Benign", "Healthy"]:
        five_non.extend(collect_images(five_dir / split / c))

all_paths_set = set()
all_cancer, all_non = [], []
for path, label in zip(binary_train_paths, binary_train_labels):
    if path not in all_paths_set:
        all_paths_set.add(path)
        (all_cancer if label == 1 else all_non).append(path)
for p in five_cancer:
    if p not in all_paths_set:
        all_paths_set.add(p)
        all_cancer.append(p)
for p in five_non:
    if p not in all_paths_set:
        all_paths_set.add(p)
        all_non.append(p)

all_paths = all_cancer + all_non
all_labels = [1] * len(all_cancer) + [0] * len(all_non)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=CONFIG["val_split"],
    random_state=CONFIG["seed"],
    stratify=all_labels,
)

print(f"Train: {len(train_paths)}, Val: {len(val_paths)}")
print(f"TM: {len(TM)}")

Building frozen test sets v2...
  5-class dataset: /Users/alokdwivedi/dev/oral_cavity_5class_dataset
  Binary dataset: /Users/alokdwivedi/dev/oral_cavity_binary_dataset
  Small binary dataset: /Users/alokdwivedi/dev/oral-images-dataset/Oral-Image-small-binary-dataset

Frozen test sets v2 saved to /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/frozen_test_sets
  T5: 534 samples (5-class test)
  TB: 246 samples (binary dataset test)
  TC: 49 samples (small binary dataset test)
  TM: 779 samples (mixed, all sources)
  binary_train_val_for_training: 1388 paths
  small_binary_train_val_for_training: 274 paths
Train: 3825, Val: 675
TM: 779


## 4. Augmentation and dataloaders

In [4]:
def get_augmentation_pipeline(mode="medical", image_size=(224, 224)):
    if HAS_ALB and mode == "medical":
        return A.Compose([
            A.Resize(image_size[0], image_size[1]),
            A.RandomRotate90(p=0.4),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
            A.HueSaturationValue(hue_shift_limit=12, sat_shift_limit=15, val_shift_limit=12, p=0.4),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    return None

def get_val_transform(image_size=(224, 224)):
    if HAS_ALB:
        return A.Compose([
            A.Resize(image_size[0], image_size[1]),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    return None

basic_train = transforms.Compose([
    transforms.Resize(CONFIG["image_size"]),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
basic_val = transforms.Compose([
    transforms.Resize(CONFIG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class BinaryDataset(Dataset):
    def __init__(self, paths, labels, albu_transform=None, torch_transform=None):
        self.paths, self.labels = paths, labels
        self.albu_transform, self.torch_transform = albu_transform, torch_transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = Image.open(self.paths[idx])
        if img.mode in ("P", "PA"):
            img = img.convert("RGBA").convert("RGB")
        else:
            img = img.convert("RGB")
        label = self.labels[idx]
        if self.albu_transform is not None:
            img = self.albu_transform(image=np.array(img))["image"]
        elif self.torch_transform is not None:
            img = self.torch_transform(img)
        return img, label

train_aug = get_augmentation_pipeline(CONFIG["augmentation"], CONFIG["image_size"])
val_aug = get_val_transform(CONFIG["image_size"])
train_ds = BinaryDataset(train_paths, train_labels, albu_transform=train_aug) if train_aug else BinaryDataset(train_paths, train_labels, torch_transform=basic_train)
val_ds = BinaryDataset(val_paths, val_labels, albu_transform=val_aug) if val_aug else BinaryDataset(val_paths, val_labels, torch_transform=basic_val)
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=CONFIG["num_workers"])
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=CONFIG["num_workers"])
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 120, Val batches: 22


/Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/venv_pytorch/lib/python3.11/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


## 5. Model and training helpers

In [5]:
class ConservativeMobileNetV3(nn.Module):
    def __init__(self, num_classes=2, dropout_rate=0.2, backbone_ckpt_path=None):
        super().__init__()
        try:
            self.backbone = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        except AttributeError:
            self.backbone = models.mobilenet_v3_large(pretrained=True)
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Dropout(dropout_rate),
            nn.Linear(960, 256), nn.ReLU(inplace=True), nn.Linear(256, num_classes),
        )
        if backbone_ckpt_path and Path(backbone_ckpt_path).exists():
            state = torch.load(backbone_ckpt_path, map_location="cpu")
            self.load_state_dict(state, strict=False)

    def forward(self, x):
        x = self.backbone.features(x)
        return self.classifier(x)

    def freeze_backbone(self):
        for p in self.backbone.features.parameters():
            p.requires_grad = False

    def unfreeze_last_k_blocks(self, k):
        children = list(self.backbone.features.children())
        for layer in children[-k:]:
            for p in layer.parameters():
                p.requires_grad = True

    def get_param_groups(self, lr_head, lr_backbone=None):
        groups = [{"params": list(self.classifier.parameters()), "lr": lr_head}]
        backbone_trainable = [p for p in self.backbone.features.parameters() if p.requires_grad]
        if backbone_trainable and lr_backbone is not None:
            groups.append({"params": backbone_trainable, "lr": lr_backbone})
        return groups


classes = np.unique(train_labels)
weights = compute_class_weight("balanced", classes=classes, y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float32)
if CONFIG.get("cancer_weight_multiplier", 1.0) != 1.0:
    class_weights = class_weights.clone()
    class_weights[1] *= CONFIG["cancer_weight_multiplier"]

def _make_criterion():
    return nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=CONFIG.get("label_smoothing", 0.0))

def _train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for data, target in train_loader:
        data, target = data.to(device), target.to(device).long()
        optimizer.zero_grad()
        out = model(data)
        loss = criterion(out, target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total += target.size(0)
        correct += out.argmax(1).eq(target).sum().item()
    return total_loss / len(train_loader), 100.0 * correct / total

def _validate(model, val_loader, criterion, device):
    model.eval()
    correct, total = 0, 0
    val_loss = 0.0
    probs_list, targets_list = [], []
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device).long()
            out = model(data)
            loss = criterion(out, target)
            val_loss += loss.item()
            probs = torch.softmax(out, dim=1)
            total += target.size(0)
            correct += out.argmax(1).eq(target).sum().item()
            probs_list.extend(probs[:, 1].cpu().numpy())
            targets_list.extend(target.cpu().numpy())
    acc = 100.0 * correct / total
    auc = roc_auc_score(targets_list, probs_list)
    return acc, auc, val_loss / len(val_loader)

## 6. Train progressive_K5 (one model)

In [6]:
def train_progressive_K5(dropout_rate, seed, train_loader, val_loader, device, tb_log_dir=None):
    warmup_epochs = CONFIG["warmup_epochs"]
    main_epochs = CONFIG["num_epochs"]
    k = CONFIG["unfreeze_last_k"]
    lr_head, lr_backbone = CONFIG["lr_head"], CONFIG["lr_backbone"]
    writer = SummaryWriter(log_dir=tb_log_dir) if (HAS_TB and tb_log_dir) else None

    model = ConservativeMobileNetV3(num_classes=NUM_CLASSES, dropout_rate=dropout_rate, backbone_ckpt_path=BACKBONE_CKPT).to(device)
    model.freeze_backbone()
    criterion = _make_criterion()
    optimizer = optim.Adam(model.classifier.parameters(), lr=lr_head)

    for epoch in range(warmup_epochs):
        _train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, val_auc, _ = _validate(model, val_loader, criterion, device)
        if epoch == warmup_epochs - 1:
            print(f"  Warmup done: val_auc={val_auc:.4f}")

    model.unfreeze_last_k_blocks(k)
    param_groups = model.get_param_groups(lr_head, lr_backbone)
    optimizer = optim.Adam(param_groups)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=main_epochs, eta_min=1e-6)

    best_auc, best_acc, best_state, patience_ctr = 0.0, 0.0, None, 0
    for epoch in range(main_epochs):
        avg_loss, train_acc = _train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_acc, val_auc, _ = _validate(model, val_loader, criterion, device)
        if writer:
            writer.add_scalar("loss/train", avg_loss, warmup_epochs + epoch)
            writer.add_scalar("acc/val", val_acc, warmup_epochs + epoch)
            writer.add_scalar("auc/val", val_auc, warmup_epochs + epoch)
        if val_auc > best_auc:
            best_auc, best_acc = val_auc, val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Ep {warmup_epochs + epoch + 1:3d}  loss={avg_loss:.4f}  val_acc={val_acc:.2f}%  val_auc={val_auc:.4f}")
        if patience_ctr >= CONFIG["patience"]:
            print(f"  Early stop at epoch {warmup_epochs + epoch + 1}")
            break
        scheduler.step()
    if writer:
        writer.close()
    if best_state:
        model.load_state_dict(best_state)
    return model, best_acc, best_auc

## 7. Train 3-model ensemble

In [7]:
models_dir = OUTPUT_DIR / "models"
tb_dir = OUTPUT_DIR / "tb_logs"
tb_dir.mkdir(exist_ok=True)
base_seed = CONFIG["seed"]
val_aucs = []

for i in range(3):
    print(f"\n--- [{i+1}/3] ensemble_model_{i+1} (dropout={DROPOUT_RATES[i]}, seed={base_seed + i}) ---")
    torch.manual_seed(base_seed + i)
    np.random.seed(base_seed + i)
    model, best_acc, best_auc = train_progressive_K5(
        DROPOUT_RATES[i], base_seed + i, train_loader, val_loader, device,
        tb_log_dir=str(tb_dir / f"ensemble_{i+1}"),
    )
    ckpt_path = models_dir / f"ensemble_model_{i+1}.pth"
    torch.save(model.state_dict(), ckpt_path)
    val_aucs.append(best_auc)
    print(f"  Saved: {ckpt_path}  val_auc={best_auc:.4f}")

print(f"\nEnsemble mean val AUC: {np.mean(val_aucs):.4f}")
with open(OUTPUT_DIR / "ensemble_val_aucs.json", "w") as f:
    json.dump({"val_aucs": val_aucs, "mean": float(np.mean(val_aucs))}, f, indent=2)


--- [1/3] ensemble_model_1 (dropout=0.3, seed=42) ---
  Warmup done: val_auc=0.8135
  Ep   6  loss=0.5179  val_acc=72.44%  val_auc=0.8435
  Ep  10  loss=0.3714  val_acc=76.00%  val_auc=0.8651
  Ep  15  loss=0.2686  val_acc=78.81%  val_auc=0.8778
  Ep  20  loss=0.2180  val_acc=77.04%  val_auc=0.8676
  Early stop at epoch 23
  Saved: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_progressive_K5_ensemble/histology_progressive_K5_ensemble_20260219_083432/models/ensemble_model_1.pth  val_auc=0.8852

--- [2/3] ensemble_model_2 (dropout=0.1, seed=43) ---
  Warmup done: val_auc=0.8156
  Ep   6  loss=0.5007  val_acc=73.78%  val_auc=0.8281
  Ep  10  loss=0.3524  val_acc=74.07%  val_auc=0.8721
  Ep  15  loss=0.2680  val_acc=78.81%  val_auc=0.8781
  Ep  20  loss=0.2091  val_acc=79.85%  val_auc=0.8793
  Ep  25  loss=0.1851  val_acc=79.56%  val_auc=0.8820
  Ep  30  loss=0.1731  val_acc=80.15%  val_auc=0.8827
  Ep  35  loss=0.1675  val_acc=81.04%

## 8. Export to ONNX

In [8]:
onnx_dir = OUTPUT_DIR / "onnx"
onnx_dir.mkdir(exist_ok=True)

def export_to_onnx(model, onnx_path, device):
    model.eval()
    model.to(device)
    dummy = torch.randn(1, 3, 224, 224).to(device)
    torch.onnx.export(
        model, dummy, str(onnx_path),
        export_params=True, opset_version=18, do_constant_folding=True,
        input_names=["input"], output_names=["output"],
        dynamic_axes={"input": {0: "batch_size"}, "output": {0: "batch_size"}},
    )
    return onnx_path.stat().st_size / 1024 / 1024

for i in range(3):
    ckpt_path = models_dir / f"ensemble_model_{i+1}.pth"
    onnx_path = onnx_dir / f"ensemble_model_{i+1}.onnx"
    model = ConservativeMobileNetV3(num_classes=NUM_CLASSES, dropout_rate=DROPOUT_RATES[i]).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    size_mb = export_to_onnx(model, onnx_path, device)
    print(f"Exported: {onnx_path.name} ({size_mb:.2f} MB)")
print(f"ONNX dir: {onnx_dir}")

/var/folders/7w/rpdt250519x332ww6xcspbnh0000gp/T/ipykernel_3232/1670027740.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0219 16:34:32.665000 3232 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 16:34:32.666000 3232 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 16:34:32.666000 3232 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 

[torch.onnx] Obtain model graph for `ConservativeMobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ConservativeMobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.
Exported: ensemble_model_1.onnx (0.39 MB)


/var/folders/7w/rpdt250519x332ww6xcspbnh0000gp/T/ipykernel_3232/1670027740.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0219 16:34:36.216000 3232 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 16:34:36.217000 3232 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 16:34:36.217000 3232 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 

...

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.
Exported: ensemble_model_2.onnx (0.39 MB)


/var/folders/7w/rpdt250519x332ww6xcspbnh0000gp/T/ipykernel_3232/1670027740.py:8: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0219 16:34:39.237000 3232 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 16:34:39.238000 3232 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0219 16:34:39.238000 3232 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 

[torch.onnx] Obtain model graph for `ConservativeMobileNetV3([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ConservativeMobileNetV3([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 93 of general pattern rewrite rules.
Exported: ensemble_model_3.onnx (0.39 MB)
ONNX dir: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_progressive_K5_ensemble/histology_progressive_K5_ensemble_20260219_083432/onnx


## 9. Evaluate on frozen sets (T5, TB, TM and TC if present)

In [9]:
eval_script = NOTEBOOK_DIR / "evaluate_on_frozen_sets.py"
eval_output = OUTPUT_DIR / "frozen_eval.json"
cmd = [
    sys.executable,
    str(eval_script),
    "--model", str(onnx_dir),
    "--model_type", "binary_onnx_ensemble",
    "--frozen_dir", str(FROZEN_DIR),
    "--output", str(eval_output),
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(NOTEBOOK_DIR))
if result.stdout:
    print(result.stdout)
if result.returncode != 0 and result.stderr:
    print("STDERR:", result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"evaluate_on_frozen_sets.py exited with {result.returncode}")

Running: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/venv_pytorch/bin/python3.11 /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/evaluate_on_frozen_sets.py --model /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_progressive_K5_ensemble/histology_progressive_K5_ensemble_20260219_083432/onnx --model_type binary_onnx_ensemble --frozen_dir /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/frozen_test_sets --output /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_progressive_K5_ensemble/histology_progressive_K5_ensemble_20260219_083432/frozen_eval.json
Running on T5...
Running on TB...
Running on TC...
Running on TM...

Frozen test sets: T5, TB, TC, TM
T5 (5-class test):  n=484  Acc=76.65%  Sens=84.90%  Spec=68.20%  AUROC=0.8635
TB (binary test):   n=246  Acc=95.12%  Sens=93.98%  Spec=96.46% 

## 10. Summary: T5, TB, TM (and TC)

In [10]:
with open(eval_output) as f:
    ev = json.load(f)

def fmt_m(name, m):
    if not m or "error" in str(m):
        return f"{name}: (no data)"
    acc = m.get("accuracy", 0) or 0
    sens = m.get("sensitivity", 0) or 0
    spec = m.get("specificity", 0) or 0
    auroc = m.get("auroc", 0) or 0
    s90 = m.get("sens_at_90spec", 0) or 0
    brier = m.get("brier", 0) or 0
    ece = m.get("ece", 0) or 0
    n = m.get("n", 0)
    return f"{name}: n={n}  Acc={acc*100:.1f}%  Sens={sens*100:.1f}%  Spec={spec*100:.1f}%  AUROC={auroc:.4f}  S@90Sp={s90*100:.1f}%  Brier={brier:.4f}  ECE={ece:.4f}"

print("Histology progressive_K5 ensemble — frozen set results")
print("=" * 90)
for name in ["T5", "TB", "TC", "TM"]:
    if name in ev:
        print(fmt_m(name, ev[name]))
print("=" * 90)
print(f"Results saved: {eval_output}")

Histology progressive_K5 ensemble — frozen set results
T5: n=484  Acc=76.7%  Sens=84.9%  Spec=68.2%  AUROC=0.8635  S@90Sp=64.1%  Brier=0.1631  ECE=0.3430
TB: n=246  Acc=95.1%  Sens=94.0%  Spec=96.5%  AUROC=0.9896  S@90Sp=96.2%  Brier=0.0382  ECE=0.4422
TC: n=49  Acc=85.7%  Sens=100.0%  Spec=72.0%  AUROC=0.9950  S@90Sp=95.8%  Brier=0.0971  ECE=0.3916
TM: n=779  Acc=83.1%  Sens=88.8%  Spec=76.9%  AUROC=0.9218  S@90Sp=76.4%  Brier=0.1195  ECE=0.3665
Results saved: /Users/alokdwivedi/dev/github/3amwave/oral-cavity-classifier/ml/training/notebooks/results/histology_progressive_K5_ensemble/histology_progressive_K5_ensemble_20260219_083432/frozen_eval.json
